# Run Metrics History

Visualises longitudinal performance recorded in `runs/metrics_log.csv`.
Each row is one execution of the `microstructure_demo` notebook.

**Sections**
1. Load & inspect the log
2. R² trend across runs (Cycle 1 avg + per-target)
3. MAE / RMSE trends
4. CV R² vs test R² (overfitting check)
5. Feature stream dimensions over time
6. Morphological coverage (rows matched)
7. Best-model frequency
8. Cycle 1 vs Cycle 1+2 comparison
9. Per-tag performance breakdown
10. Summary table (latest 10 runs)


In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

# Allow imports from the model src package
sys.path.insert(0, os.path.abspath('..'))

from src.metrics_logger import load_log

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 10,
})

LOG_PATH = os.path.abspath('../runs/metrics_log.csv')
print(f'Log path: {LOG_PATH}')
print(f'Exists:   {os.path.exists(LOG_PATH)}')

## 1. Load & Inspect

In [ ]:
df = load_log(LOG_PATH)

if df.empty:
    print('No runs logged yet.  Execute the microstructure_demo notebook first.')
else:
    # Parse timestamps; add a sequential run index for plotting
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.sort_values('timestamp').reset_index(drop=True)
    df['run'] = df.index + 1

    # Coerce numeric columns
    _numeric = [
        'n_samples', 'n_features',
        'n_image_features', 'n_morph_features', 'n_tabular_features',
        'morph_rows_matched',
        'c1_test_r2_avg', 'c1_test_mae_avg', 'c1_test_rmse_avg',
        'c1_holdingtemp_r2', 'c1_holdingtemp_mae', 'c1_holdingtemp_rmse',
        'c1_holdingtime_r2', 'c1_holdingtime_mae', 'c1_holdingtime_rmse',
        'c1_cv_r2_mean', 'c1_cv_r2_std',
        'c1c2_test_r2_avg', 'c1c2_test_mae_avg', 'c1c2_test_rmse_avg',
    ]
    for c in _numeric:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')

    print(f'Runs loaded: {len(df)}')
    print(f'Date range : {df["timestamp"].min()}  →  {df["timestamp"].max()}')
    print(f'Columns    : {list(df.columns)}')
    df.head()

## 2. R² Trend

In [ ]:
if len(df) == 0:
    print('No data.')
else:
    fig, ax = plt.subplots(figsize=(10, 4))

    x = df['run']
    ax.plot(x, df['c1_test_r2_avg'], 'o-', color='steelblue', linewidth=2,
            markersize=6, label='C1 avg R²', zorder=3)

    if 'c1_holdingtemp_r2' in df.columns:
        ax.plot(x, df['c1_holdingtemp_r2'], 's--', color='tomato', linewidth=1.4,
                markersize=5, label='HoldingTemp R²', alpha=0.85)
    if 'c1_holdingtime_r2' in df.columns:
        ax.plot(x, df['c1_holdingtime_r2'], '^--', color='seagreen', linewidth=1.4,
                markersize=5, label='HoldingTime R²', alpha=0.85)
    if 'c1c2_test_r2_avg' in df.columns and df['c1c2_test_r2_avg'].notna().any():
        ax.plot(x, df['c1c2_test_r2_avg'], 'D:', color='goldenrod', linewidth=1.4,
                markersize=5, label='C1+C2 avg R²', alpha=0.85)

    # Annotate best run
    best_idx = df['c1_test_r2_avg'].idxmax()
    best_run = df.loc[best_idx]
    ax.annotate(
        f"best\n{best_run['c1_test_r2_avg']:.3f}",
        xy=(best_run['run'], best_run['c1_test_r2_avg']),
        xytext=(8, 8), textcoords='offset points',
        fontsize=8, color='steelblue',
        arrowprops=dict(arrowstyle='->', color='steelblue', lw=1)
    )

    ax.axhline(0, color='grey', linewidth=0.8, linestyle=':')
    ax.set_xlabel('Run')
    ax.set_ylabel('R²')
    ax.set_title('Test R² across runs')
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    ax.legend(loc='lower right')
    plt.tight_layout()
    plt.show()

## 3. MAE & RMSE Trends

In [ ]:
if len(df) == 0:
    print('No data.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    for ax, metric, label, color in [
        (axes[0], 'mae', 'MAE', 'steelblue'),
        (axes[1], 'rmse', 'RMSE', 'tomato'),
    ]:
        x = df['run']
        col_avg  = f'c1_test_{metric}_avg'
        col_temp = f'c1_holdingtemp_{metric}'
        col_time = f'c1_holdingtime_{metric}'

        if col_avg in df.columns:
            ax.plot(x, df[col_avg], 'o-', color=color, linewidth=2,
                    markersize=6, label=f'C1 avg {label}', zorder=3)
        if col_temp in df.columns:
            ax.plot(x, df[col_temp], 's--', color='darkorange', linewidth=1.4,
                    markersize=5, label=f'HoldingTemp {label}', alpha=0.85)
        if col_time in df.columns:
            ax.plot(x, df[col_time], '^--', color='seagreen', linewidth=1.4,
                    markersize=5, label=f'HoldingTime {label}', alpha=0.85)

        ax.set_xlabel('Run')
        ax.set_ylabel(label)
        ax.set_title(f'Test {label} across runs')
        ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
        ax.legend(fontsize=8)

    plt.tight_layout()
    plt.show()

## 4. CV R² vs Test R² — Overfitting Check

In [ ]:
if len(df) == 0 or 'c1_cv_r2_mean' not in df.columns or df['c1_cv_r2_mean'].isna().all():
    print('No CV data logged yet.')
else:
    mask = df['c1_cv_r2_mean'].notna()
    sub = df[mask]

    fig, ax = plt.subplots(figsize=(10, 4))
    x = sub['run']

    ax.plot(x, sub['c1_test_r2_avg'], 'o-', color='steelblue', linewidth=2,
            markersize=6, label='Test R²', zorder=3)
    ax.plot(x, sub['c1_cv_r2_mean'], 's-', color='darkorange', linewidth=2,
            markersize=6, label='CV R² (mean)', zorder=3)

    if 'c1_cv_r2_std' in sub.columns:
        ax.fill_between(
            x,
            sub['c1_cv_r2_mean'] - sub['c1_cv_r2_std'],
            sub['c1_cv_r2_mean'] + sub['c1_cv_r2_std'],
            alpha=0.15, color='darkorange', label='CV ±1 std'
        )

    # Gap annotation on the latest run
    last = sub.iloc[-1]
    gap = last['c1_test_r2_avg'] - last['c1_cv_r2_mean']
    ax.annotate(
        f'gap={gap:+.3f}',
        xy=(last['run'], (last['c1_test_r2_avg'] + last['c1_cv_r2_mean']) / 2),
        xytext=(10, 0), textcoords='offset points',
        fontsize=8, color='grey'
    )

    ax.axhline(0, color='grey', linewidth=0.8, linestyle=':')
    ax.set_xlabel('Run')
    ax.set_ylabel('R²')
    ax.set_title('CV R² vs Test R² (positive gap = test > CV → possible overfitting)')
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    ax.legend()
    plt.tight_layout()
    plt.show()

## 5. Feature Stream Dimensions over Time

In [ ]:
_stream_cols = ['n_image_features', 'n_morph_features', 'n_tabular_features']
_present = [c for c in _stream_cols if c in df.columns and df[c].notna().any()]

if len(df) == 0 or not _present:
    print('No feature-dimension data logged yet.')
else:
    fig, axes = plt.subplots(1, len(_present), figsize=(4 * len(_present), 4), squeeze=False)
    colors = {'n_image_features': 'steelblue',
              'n_morph_features': 'seagreen',
              'n_tabular_features': 'tomato'}
    labels = {'n_image_features': 'Image (CNN)',
               'n_morph_features': 'Morphological',
               'n_tabular_features': 'Tabular'}

    for ax, col in zip(axes[0], _present):
        sub = df[df[col].notna()]
        ax.bar(sub['run'], sub[col], color=colors.get(col, 'grey'), alpha=0.8)
        ax.set_xlabel('Run')
        ax.set_ylabel('Dimensions')
        ax.set_title(labels.get(col, col))
        ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

    plt.suptitle('Feature stream dimensions per run', y=1.02)
    plt.tight_layout()
    plt.show()

## 6. Morphological Coverage

In [ ]:
if (len(df) == 0
        or 'morph_rows_matched' not in df.columns
        or df['morph_rows_matched'].isna().all()):
    print('No morphological coverage data logged yet.')
else:
    sub = df[df['morph_rows_matched'].notna()].copy()
    sub['morph_pct'] = 100.0 * sub['morph_rows_matched'] / sub['n_samples']

    fig, ax = plt.subplots(figsize=(10, 4))
    x = sub['run']

    bars = ax.bar(x, sub['morph_pct'], color='seagreen', alpha=0.75, zorder=3)
    ax.axhline(100, color='grey', linewidth=0.8, linestyle=':')

    for bar, val in zip(bars, sub['morph_pct']):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                f'{val:.0f}%', ha='center', va='bottom', fontsize=8)

    ax.set_xlabel('Run')
    ax.set_ylabel('Rows with images (%)')
    ax.set_title('Morphological image coverage per run')
    ax.set_ylim(0, 115)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    plt.tight_layout()
    plt.show()

## 7. Best-Model Frequency

In [ ]:
if len(df) == 0 or 'c1_best_model' not in df.columns:
    print('No data.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    for ax, col, title in [
        (axes[0], 'c1_best_model', 'Cycle 1 — best model'),
        (axes[1], 'c1c2_best_model', 'Cycle 1+2 — best model'),
    ]:
        if col not in df.columns or df[col].isna().all():
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(title)
            continue
        counts = df[col].dropna().value_counts()
        ax.barh(counts.index, counts.values, color='steelblue', alpha=0.8)
        ax.set_xlabel('Times selected as best')
        ax.set_title(title)
        ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

    plt.tight_layout()
    plt.show()

## 8. Cycle 1 vs Cycle 1+2 R² Comparison

In [ ]:
if (len(df) == 0
        or 'c1c2_test_r2_avg' not in df.columns
        or df['c1c2_test_r2_avg'].isna().all()):
    print('No Cycle 1+2 data logged yet.')
else:
    sub = df[df['c1c2_test_r2_avg'].notna()]
    x = sub['run']
    width = 0.35

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(x - width / 2, sub['c1_test_r2_avg'], width,
           label='Cycle 1 (2 targets)', color='steelblue', alpha=0.85)
    ax.bar(x + width / 2, sub['c1c2_test_r2_avg'], width,
           label='Cycle 1+2 (4 targets)', color='goldenrod', alpha=0.85)

    ax.axhline(0, color='grey', linewidth=0.8, linestyle=':')
    ax.set_xlabel('Run')
    ax.set_ylabel('Test R² (avg over targets)')
    ax.set_title('Cycle 1 vs Cycle 1+2 test R²')
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    ax.legend()
    plt.tight_layout()
    plt.show()

## 9. Per-Tag Performance Breakdown

Groups runs by tags and shows the distribution of test R².
Only shown when at least one run has a non-empty tag.

In [ ]:
if len(df) == 0 or 'tags' not in df.columns:
    print('No data.')
else:
    # Explode multi-tag rows (tags are '; '-separated)
    df_tags = df.copy()
    df_tags['tag_list'] = df_tags['tags'].fillna('').apply(
        lambda s: [t.strip() for t in s.split(';') if t.strip()] or ['(untagged)']
    )
    exploded = df_tags.explode('tag_list')

    tagged_only = exploded[exploded['tag_list'] != '(untagged)']
    if tagged_only.empty:
        print('No tagged runs yet.  Use m.add_tag("...") in cell 52 to annotate runs.')
    else:
        grouped = (
            tagged_only
            .groupby('tag_list')['c1_test_r2_avg']
            .agg(['mean', 'std', 'count'])
            .sort_values('mean', ascending=False)
        )

        fig, ax = plt.subplots(figsize=(max(6, len(grouped) * 1.2), 4))
        x = np.arange(len(grouped))
        ax.bar(x, grouped['mean'], yerr=grouped['std'].fillna(0),
               color='steelblue', alpha=0.8, capsize=4,
               error_kw=dict(elinewidth=1.2, ecolor='grey'))
        ax.set_xticks(x)
        ax.set_xticklabels(grouped.index, rotation=30, ha='right')
        ax.set_ylabel('C1 Test R² (mean ± std)')
        ax.set_title('Average test R² by tag')

        for xi, (mean, count) in enumerate(zip(grouped['mean'], grouped['count'])):
            ax.text(xi, mean + 0.01, f'n={count}', ha='center', fontsize=8)

        plt.tight_layout()
        plt.show()
        display(grouped.rename(columns={'mean': 'r2_mean', 'std': 'r2_std', 'count': 'n_runs'}))

## 10. Summary Table — Latest 10 Runs

In [ ]:
if len(df) == 0:
    print('No runs logged yet.')
else:
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 160)
    pd.set_option('display.float_format', '{:.4f}'.format)

    view_cols = [
        'run', 'timestamp', 'git_commit', 'backbone',
        'n_samples', 'n_features',
        'n_image_features', 'n_morph_features', 'n_tabular_features', 'morph_rows_matched',
        'c1_best_model',
        'c1_test_r2_avg', 'c1_test_mae_avg', 'c1_test_rmse_avg',
        'c1_holdingtemp_r2', 'c1_holdingtime_r2',
        'c1_cv_r2_mean', 'c1_cv_r2_std',
        'c1c2_test_r2_avg',
        'tags',
    ]
    present = [c for c in view_cols if c in df.columns]
    display(df[present].tail(10).reset_index(drop=True))